# B3 — mixed displacement-stress formulation: REAL training run

**Follow-up paper, Professor Rabczuk's item 1** (2026-10-16/18/20 emails): the network predicts both `u` and the first Piola-Kirchhoff stress `P`, trained against constitutive consistency (`P = dpsi/dF`) and equilibrium (`Div_0(P) = 0`) instead of total potential energy. Kept entirely separate from `train_B3.py` (the DEM baseline that produced the submitted paper's own published B3 numbers).

**Why this is cleared to run now, not another diagnostic**: two prior checks already passed --
1. 5 closed-form correctness checks (constitutive vs. autograd, divergence operator on two known fields, affine-field end-to-end), all at 1e-10 or tighter, on CPU and re-confirmed in this exact Colab environment.
2. A 300-iteration fixed-batch diagnostic AT PRODUCTION MODEL SIZE on a real A100 showed the loss is stable (no NaN/divergence) and BOTH the constitutive and equilibrium residuals genuinely decrease (equilibrium converged to ~0.0002, constitutive dropped 428,186 -> 51,092) at `w_equilibrium=28000`.

Neither of those was a real training run (new random samples every iteration, thousands of iterations, a checkpoint meant to be evaluated). This cell is that run: `n_iters=50000`, `batch_size=8` -- the exact same budget `train_B3.py`'s own baseline used, so the two checkpoints are directly comparable.

**What happens in this cell**: train for 50,000 iterations (expect roughly 4-8 hours based on the diagnostic's own per-iteration cost being heavier than the baseline DEM loss's 0.15s/iteration -- watch the printed s/iteration rate in the first few hundred iterations and compare), save checkpoints to Drive every 5,000 iterations, then immediately evaluate the final checkpoint with `evaluate_B3_mixed_stress.py` against the same held-out FEM dataset the baseline was evaluated on -- reporting both the "implicit" stress (autograd from the predicted `u`, same as the baseline) and the new "direct" stress (the network's own predicted `P`), so the printed output directly answers whether predicting stress explicitly actually helps the stress QoI. A `run_manifest.json` entry is written at the end (Professor Rabczuk's own "please save the results").

In [ ]:
# =====================================================================
#  CELL -- B3 mixed displacement-stress: REAL training run (50,000
#  iterations, production model size), then evaluation against the
#  held-out FEM dataset. See the markdown cell above for why this is
#  cleared to run now.
# =====================================================================
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import subprocess
import sys
import time

_started = time.time()


def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted at /content/drive.')
else:
    drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import torch
assert torch.cuda.is_available(), 'this cell needs a real GPU'
print('GPU:', torch.cuda.get_device_name(0))

import gc
for _attr in ('last_traceback', 'last_value', 'last_type'):
    if hasattr(sys, _attr):
        delattr(sys, _attr)
gc.collect()
torch.cuda.empty_cache()
print(f'GPU memory at start: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, '
      f'{torch.cuda.memory_reserved()/1e9:.2f} GB reserved (should be ~0 -- if not, '
      'use Runtime > Restart session before trusting this run)')

from omar_pfem.train_B3_mixed_stress import get_args, train
from omar_pfem.data.data_generate_B3_dataset import DEFAULT_RESOLUTION

device = torch.device('cuda')

R = '/content/drive/MyDrive/pfem_run'
OUTPUT_DIR = f'{R}/b3_mixed_stress_training'
os.makedirs(OUTPUT_DIR, exist_ok=True)

args = get_args([
    '--n_iters', '50000',
    '--batch_size', '8',
    '--log_every', '500',
    '--ckpt_every', '5000',
    '--output_dir', OUTPUT_DIR,
    # w_equilibrium=28000 and p_output_scale=500.0 are this script's own
    # defaults, confirmed stable at production scale by the diagnostic
    # notebook -- not overridden here.
])
n_elem = (DEFAULT_RESOLUTION[0] - 1) * (DEFAULT_RESOLUTION[1] - 1) * (DEFAULT_RESOLUTION[2] - 1)
print(f'\nTraining at {DEFAULT_RESOLUTION} -> {n_elem} elements, {args.n_iters} iterations, '
      f'batch_size={args.batch_size}, w_constitutive={args.w_constitutive}, '
      f'w_equilibrium={args.w_equilibrium}, p_output_scale={args.p_output_scale}, '
      f'checkpoints -> {OUTPUT_DIR}')
print('Watch the s/iteration rate in the first few hundred iterations -- if it is wildly '
      'different from what the production diagnostic implied, interrupt and report the '
      'observed rate rather than letting a possibly-broken run continue for hours.')

torch.cuda.reset_peak_memory_stats(device)
t0 = time.time()
model = train(args, device)
elapsed_total = time.time() - t0

print(f'\n{"=" * 90}')
print(f'Training finished in {elapsed_total:.1f}s ({elapsed_total / args.n_iters:.4f}s/iteration average).')
print(f'GPU peak memory during training: '
      f'{torch.cuda.max_memory_allocated(device) / 1e6:.1f}MB allocated, '
      f'{torch.cuda.max_memory_reserved(device) / 1e6:.1f}MB reserved.')

# =====================================================================
#  Evaluation: same held-out FEM dataset the baseline DEM checkpoint was
#  evaluated on, so the two sets of numbers are directly comparable.
# =====================================================================
print(f'\n{"=" * 90}')
print('=== Evaluating the final checkpoint against the held-out FEM dataset ===')
CHECKPOINT = f'{OUTPUT_DIR}/checkpoint_{args.n_iters}.pt'
DATASET = f'{R}/b3_dataset_clean_holdout/dataset.h5'
OUT_JSON = f'{OUTPUT_DIR}/qois_{args.n_iters}.json'

sys.argv = [
    'evaluate_B3_mixed_stress.py',
    '--checkpoint', CHECKPOINT,
    '--dataset', DATASET,
    '--out_json', OUT_JSON,
]
from omar_pfem.evaluate_B3_mixed_stress import main as evaluate_main
evaluate_main()

try:
    from omar_pfem.run_manifest import write_manifest
    write_manifest(OUTPUT_DIR, kind='b3_mixed_stress_training',
                    args={'n_iters': args.n_iters, 'batch_size': args.batch_size,
                          'resolution': DEFAULT_RESOLUTION, 'lr': args.lr,
                          'w_constitutive': args.w_constitutive,
                          'w_equilibrium': args.w_equilibrium,
                          'p_output_scale': args.p_output_scale},
                    started_at=_started,
                    results={'elapsed_total_s': elapsed_total,
                             's_per_iter': elapsed_total / args.n_iters},
                    outputs=[CHECKPOINT, OUT_JSON],
                    notes="First real training run of the mixed displacement-stress "
                          "formulation (Professor Rabczuk's item 1, 2026-10-16/18/20 "
                          "emails) -- network predicts u and P jointly, trained on "
                          "constitutive consistency + equilibrium residuals instead of "
                          "total potential energy. Preceded by 5 CPU correctness checks "
                          "and a production-scale fixed-batch diagnostic (both passing) "
                          "before any real training time was spent. See "
                          "evaluate_B3_mixed_stress.py's own output/qois_*.json for "
                          "whether the direct stress prediction beats the implicit "
                          "(autograd-from-u) one against the held-out FEM dataset.")
except Exception as e:
    print(f'[manifest] not recorded: {e}')

print('\nDone. Send this cell\'s full printed output back, especially the "Key comparison '
      'for Professor Rabczuk\'s item 1" line near the end of the evaluation output.')
